#### 08. Tier 3 features

Tier 3 adds time and direction to the neighbour picture of Tier 2. Tier 2 says how much it is raining around the station now; Tier 3 says whether that rain is growing or easing, coming closer or moving away, and whether it is raining where the storms of the current season come from.

- Changes over 15 minutes. Twelve Tier 2 summaries (neighbour rain near and far, the wettest neighbour, the shares of wet neighbours, the distance to the nearest rain, the four compass quadrants and the island-wide values) are computed again from the neighbours' readings 15 minutes earlier, and each feature is the difference, now minus then. A negative `t3_d15_dist_rain_km`, for example, means the nearest rain has come closer.
- Rain upstream. Task 1 measured how storms move in each season (Figure 23): towards the east at 38 km/h in the southwest monsoon, the south-east at 51 km/h in the northeast monsoon and the east-south-east at 59 km/h in between. Following that motion backwards from the station gives the point the rain arriving in 15 and in 30 minutes is coming from. The features are the rain of the last 15 minutes around that point (Gaussian weights, σ = 4 km, within 10 km), the share of wet stations there, and the number of stations near the 30-minute point, which often lies over the sea.

As in Tier 2, all stations of one slot sit side by side in one row, now next to the same row 15 minutes earlier, so each feature is a small matrix product applied with `mapInPandas`. The result is saved to `task2/data/feat_t3/` and joined into the train, validation and test tables by notebook 03, so this notebook runs after 06 and before 03.

**Setup**

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../code")
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import config as C
import features as FT

spark = C.get_spark("task2-tier3")

**Building the features**

In [2]:
feat_t1 = spark.read.parquet(str(C.FEAT_T1))
FT.add_tier3(spark, feat_t1).write.mode("overwrite").partitionBy("year").parquet(str(C.FEAT_T3))

feat_t3 = spark.read.parquet(str(C.FEAT_T3))
T3 = [c for c in feat_t3.columns if c.startswith("t3_")]
print(f"{feat_t3.count():,} rows, {len(T3)} features:", T3)

48,409,778 rows, 17 features: ['t3_d15_idw_rain_30', 't3_d15_far_rain_30', 't3_d15_max_rain_30', 't3_d15_wet_frac_5km', 't3_d15_wet_frac_10km', 't3_d15_dist_rain_km', 't3_d15_rain_30_n', 't3_d15_rain_30_e', 't3_d15_rain_30_s', 't3_d15_rain_30_w', 't3_d15_island_wet_frac', 't3_d15_island_rain_30', 't3_up15_rain', 't3_up15_wet', 't3_up30_rain', 't3_up30_wet', 't3_up30_nbrs']


**The storm reaching S07**

The storm of notebook 06 (S07, 14 April 2023, every 15 minutes), seen through Tier 3.

In [3]:
cols = ["t3_d15_idw_rain_30", "t3_d15_dist_rain_km", "t3_d15_rain_30_w", "t3_up15_rain", "t3_up30_rain", "t3_up30_nbrs"]
(feat_t3.filter((F.col("station_id") == "S07") & F.col("slot").between(660850, 660880))
        .join(feat_t1.select("station_id", "slot", "t1_rain_30"), ["station_id", "slot"])
        .withColumn("time", F.date_format(F.timestamp_seconds(F.lit(C.ORIGIN_EPOCH) + F.col("slot") * C.SLOT_SECONDS), "HH:mm"))
        .orderBy("slot").select("time", "t1_rain_30", *cols).toPandas().iloc[::3].set_index("time").round(3))

,t1_rain_30,t3_d15_idw_rain_30,t3_d15_dist_rain_km,t3_d15_rain_30_w,t3_up15_rain,t3_up30_rain,t3_up30_nbrs
time,,,,,,,
14:50,0.0,0.000,0.000,0.000,0.000,0.0,0.0
15:05,0.0,0.000,0.000,0.000,0.000,0.0,0.0
15:20,0.0,0.000,0.000,0.000,0.000,0.0,0.0
15:35,0.0,0.000,0.000,0.000,0.000,0.0,0.0
15:50,0.0,0.035,-26.761,0.122,0.180,0.0,0.0
16:05,0.0,0.759,0.000,0.783,0.014,0.0,0.0
16:20,2.8,0.984,-0.614,0.087,0.000,0.0,0.0
16:35,3.2,1.362,-0.972,-0.774,0.000,0.0,0.0
16:50,0.4,-0.616,0.972,-0.209,0.000,0.0,0.0


April is inter-monsoon, when storms move east-south-east, so the upstream points lie west-north-west of S07. At 15:50, 25 minutes before S07's own rain begins, the nearest rain jumps from beyond 30 km to 3.2 km (`t3_d15_dist_rain_km` −26.8) and the 15-minute upstream point is already wet. At 16:05 the neighbours' rain is still building (+0.76 mm in 15 minutes), mostly in the west. After 16:50 every change turns negative and the nearest rain moves away again as the storm passes to the east. The 30-minute upstream point of S07 has no station within 10 km, so the two `up30` features stay 0 and `t3_up30_nbrs` says why.

**Which features carry signal**

As for Tiers 1 and 2, each feature on its own ranks a 20 % sample of the training rows by the chance of rain in the next 30 minutes (AUC-PR), on all rows and on the rows where the station is dry now. The strongest Tier 2 feature, `t2_idw_rain_15`, is shown for reference. This view cannot show what Tier 3 adds on top of Tier 2, because the two are related; the ablation in notebook 07 measures that.

In [4]:
feat_t2 = spark.read.parquet(str(C.FEAT_T2)).select("station_id", "slot", "year", "t2_idw_rain_15")
cols = T3 + ["t2_idw_rain_15"]
sample = (spark.read.parquet(str(C.LABELS))
               .filter(F.col("year").between(*C.TRAIN_YEARS))
               .sample(fraction=0.2, seed=C.SEED)
               .join(feat_t3, ["station_id", "slot", "year"])
               .join(feat_t2, ["station_id", "slot", "year"])
               .withColumn("wet", (F.col("label") > 0).cast("double"))
               .select("wet", "regime", *cols)
               .cache())
subsets = {"all": sample, "dry now": sample.filter(F.col("regime") == "dry_now")}

means = sample.groupBy("wet").agg(*[F.avg(c).alias(c) for c in cols]).toPandas().set_index("wet")
direction = {c: 1 if means.loc[1.0, c] >= means.loc[0.0, c] else -1 for c in cols}
ev = BinaryClassificationEvaluator(labelCol="wet", metricName="areaUnderPR")

rows = []
for c in cols:
    row = {"feature": c, "direction": "+" if direction[c] > 0 else "−"}
    for name, df in subsets.items():
        row[f"AUC-PR {name}"] = ev.setRawPredictionCol("s").evaluate(df.withColumn("s", F.col(c) * direction[c]))
    rows.append(row)
rates = {f"AUC-PR {name}": df.agg(F.avg("wet")).first()[0] for name, df in subsets.items()}
signal = pd.DataFrame(rows + [{"feature": "event rate (no skill)", "direction": "", **rates}]).set_index("feature")
signal.round(4)

,direction,AUC-PR all,AUC-PR dry now
feature,,,
t3_d15_idw_rain_30,+,0.4465,0.2697
t3_d15_far_rain_30,+,0.2704,0.1687
t3_d15_max_rain_30,+,0.3213,0.2188
t3_d15_wet_frac_5km,+,0.3587,0.2338
t3_d15_wet_frac_10km,+,0.3741,0.2650
t3_d15_dist_rain_km,−,0.1123,0.0706
t3_d15_rain_30_n,+,0.3080,0.1501
t3_d15_rain_30_e,+,0.3321,0.1733
t3_d15_rain_30_s,+,0.3305,0.1692


On their own, the Tier 3 features rank onsets less well than plain neighbour rain: when the station is dry now, the best of them (the 15-minute change of the distance-weighted neighbour rain, 0.27, and of the wet share within 10 km, 0.27) stay below `t2_idw_rain_15` (0.32), though all but the station count beat the event rate, most of them by five to eleven times (the change in distance to the nearest rain is the weakest, at three times). The upstream features are weaker still on their own (0.20 at 15 minutes, 0.12 at 30 minutes), as expected for a subset of the neighbours placed by an average storm motion. What matters is whether they add to Tier 2 in a model, which the T0+T1+T2 against T0+T1+T2+T3 ablation in notebook 07 measures.